In [ ]:
# ==========================================
# 1. DECORATORS & FUNCTION WRAPPERS (Selected: Q110)
# ==========================================
# Question 110:
# Write a decorator @timer that measures and prints the execution time of any function it wraps. 
# Then write a decorator @call_count that counts how many times the decorated function has been called 
# and prints the count after each call. Stack both decorators on a sample function.
#
# Sample Input:  sample_function(2) called twice
# Sample Output: 4

import time
import functools

def timer(func):
    @functools.wraps(func)
    def wrapper(n):
        s = time.time()
        r = func(n)
        t = time.time() - s
        print("Execution time:", t)
        return r
    return wrapper


def call_count(func):
    c = 0

    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        nonlocal c
        c += 1
        r = func(*args, **kwargs)
        print("Call count:", c)
        return r
    return wrapper


@timer
@call_count
def sample_function(n):
    return n * n


if __name__ == '__main__':
    sample_function(2)
    print("Q110 Output:", sample_function(2))

Call count: 1
Execution time: 0.00042057037353515625
Call count: 2
Execution time: 2.86102294921875e-05
Q110 Output: 4


In [ ]:
# ==========================================
# 2. TYPE VALIDATION DECORATORS (Selected: Q111)
# ==========================================
# Question 111:
# Write a decorator @validate_types that reads the function's type annotations and raises TypeError 
# with a descriptive message if any argument passed does not match its annotated type. Test it on a function 
# with at least four parameters of different types.
#
# Sample Input:  process_user_data("Alice", 30, 75.5, True)
# Sample Output: True

def validate_types(func):
    @functools.wraps(func)
    def wrapper(name,age,score,is_active):
        try:
            r = func(name,age,score,is_active)
            if not isinstance(name,str): raise TypeError("name : str")
            if not isinstance(age,int): raise TypeError("age : int")
            if not isinstance(score,float): raise TypeError("score : float")
            if not isinstance(is_active,bool): raise TypeError("is_active : bool")
        except TypeError as e:
            print(e)
        return r
    return wrapper
@validate_types
def process_user_data(name: str, age: int, score: float, is_active: bool) -> bool:
    return True 

if __name__ == '__main__':
    # Test Question 111
    print("Q111 Output:", process_user_data("Alice", 30, 75.5, True))


import functools

def validate_types(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):

        annotations = func.__annotations__

        for name, value in zip(func.__code__.co_varnames, args):
            expected_type = annotations.get(name)

            if expected_type and not isinstance(value, expected_type):
                raise TypeError(
                    f"{name} must be {expected_type.__name__}"
                )

        return func(*args, **kwargs)

    return wrapper


@validate_types
def process_user_data(
    name: str,
    age: int,
    score: float,
    is_active: bool
) -> bool:
    return True


if __name__ == '__main__':
    print(
        "Q111 Output:",
        process_user_data("Alice", 30, 75.5, True)
    )

Q111 Output: True
Q111 Output: True


In [15]:
# ==========================================
# 3. CLOSURES & STATE MANAGEMENT (Selected: Q112)
# ==========================================
# Question 112:
# Write a function make_counter(start=0, step=1) that returns a closure with three inner functions: 
# increment(), decrement(), and reset(). The counter state should be shared across all three. Demonstrate 
# that two independent counters do not interfere with each other.
#
# Sample Input:  c1 = make_counter(0, 1), c2 = make_counter(10, 5)
# Sample Output: {'c1_inc': 1, 'c2_inc': 15, 'c1_reset': 0, 'c2_dec': 10}

def make_counter(start=0, step=1):
    counter = start
    def inc():
        nonlocal counter
        counter += step
        return counter
    def dec():
        nonlocal counter
        counter -= step
        return counter
    def reset(): 
        counter = 0
        return counter
    return inc,dec,reset

if __name__ == '__main__':
    # Test Question 112
    c1_inc, c1_dec, c1_reset = make_counter(0, 1)
    c2_inc, c2_dec, c2_reset = make_counter(10, 5)
    
    res1 = c1_inc()
    res2 = c2_inc()
    res3 = c1_reset()
    res4 = c2_dec()
    
    print("Q112 Output:", {'c1_inc': res1, 'c2_inc': res2, 'c1_reset': res3, 'c2_dec': res4})

Q112 Output: {'c1_inc': 1, 'c2_inc': 15, 'c1_reset': 0, 'c2_dec': 10}


In [25]:
# ==========================================
# 4. FUNCTION CURRYING (Selected: Q113)
# ==========================================
# Question 113:
# Implement function currying: write a function curry(f) that takes a function of n arguments and 
# returns a curried version that can be called one argument at a time. For example, curry(add)(1)(2)(3) 
# should return 6 for add(a,b,c)=a+b+c.
#
# Sample Input:  add(a, b, c) where a=1, b=2, c=3
# Sample Output: 6

def curry(f):
    def first(a):
        def second(b):
            def third(c):
                return f(a, b, c)
            return third
        return second
    return first

def curry(f):
    def curried(*args):
        if len(args) == f.__code__.co_argcount:
            return f(*args)
    
        return lambda x: curried(*args, x)
    return curried


def add3(a, b, c):
    return a + b + c

if __name__ == '__main__':
    # Test Question 113
    curried_add = curry(add3)
    print("Q113 Output:", curried_add(1)(2)(3))


Q113 Output: 6


In [53]:
# ==========================================
# 5. GENERATORS & ITERATORS (Selected: Q114)
# ==========================================
# Question 114:
# Write a generator function flatten_gen(nested) that yields elements one at a time from a deeply 
# nested list of any depth. Then write a second generator prime_gen() that yields prime numbers indefinitely 
# using the Sieve of Eratosthenes concept. Use itertools.islice to take the first 50 primes.
#
# Sample Input:  nested = [1, [2, [3, 4], 5], 6], slice first 5 primes
# Sample Output: ([1, 2, 3, 4, 5, 6], [2, 3, 5, 7, 11])

import itertools

def flatten_gen(nested):
    for i in nested:
        if isinstance(i,list): yield from flatten_gen(i)
        else: yield i


def prime_gen():
    primes = []
    n = 2
    while True:
        is_prime = True

        for p in primes:
            if p * p > n:
                break

            if n % p == 0:
                is_prime = False
                break

        if is_prime:
            primes.append(n)
            yield n

        n += 1
if __name__ == '__main__':
    # Test Question 114
    flat_res = list(flatten_gen([1, [2, [3, 4], 5], 6]))
    primes_res = list(itertools.islice(prime_gen(), 5))
    print("Q114 Output:", (flat_res,primes_res))

Q114 Output: ([1, 2, 3, 4, 5, 6], [2, 3, 5, 7, 11])


In [57]:
# ==========================================
# 6. FUNCTION PIPELINES (Selected: Q115)
# ==========================================
# Question 115:
# Write a function pipeline(*functions) that returns a new function that passes its input through each 
# function in order (the output of one becomes the input of the next). Demonstrate a pipeline of five string
# transformation functions.
#
# Sample Input:  "  Hello World  " passed through strip, lower, replace, title, reverse
# Sample Output: 'dlroW olleH'

def pipeline(*functions):

    def action(val):
        for f in functions:
            val = f(val)
        return val
    return action
       
    return action(f)

if __name__ == '__main__':
    # Test Question 115
    p = pipeline(
        str.strip,
        str.lower,
        lambda s: s.replace("world", "python"),
        str.title,
        lambda s: s[::-1]
    )
    print("Q115 Output:", p("  Hello World  "))

Q115 Output: nohtyP olleH


In [58]:
# ==========================================
# 7. RETRY DECORATOR WITH ADVANCED ARGS (Selected: Q116)
# ==========================================
# Question 116:
# Write a decorator @retry(times=3, delay=0, exceptions=(Exception,)) that retries the decorated 
# function up to times attempts when it raises one of the specified exception types, waiting delay seconds 
# between attempts. Raise the final exception if all attempts fail. Test with a function that fails randomly.
#
# Sample Input:  unstable_func() configured with 3 retries
# Sample Output: 'Success'

import time
import functools
import random

def retry(times=3, delay=0, exceptions=(Exception,)):

    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):

            for attempt in range(times):
                try:
                    return func(*args, **kwargs)
                except exceptions:
                    if attempt == times - 1:
                        raise
                    time.sleep(delay)
        return wrapper
    return decorator

@retry(times=3, delay=0)
def unstable_func(attempts=[0]):
    attempts[0] += 1
    if attempts[0] < 3:
        raise Exception("Failed")
    return "Success"

if __name__ == '__main__':
    print("Q116 Output:", unstable_func())

Q116 Output: Success
